# 06 · Linear GAM: results and comparison

**Input:** the outputs of `scripts/linear_gam.py` (validation, final models, frozen forecast) and of `scripts/linear_gam_test_evaluation.py` (test scores, Section 5 only).
**Output:** figures `reports/figures/06_*.png`.

This notebook fits and scores nothing; it plots what the scripts wrote. Run them first:

```
python scripts/linear_gam.py
python scripts/linear_gam_test_evaluation.py
```

**Model.** Same approach as the linear regression in `04`, on the same table from `03`: one model per forecast day predicting how much warmer or colder than the historical average an hour will be, recent runs weighted more (half-life 180 days), the same 12 inputs. The only change is the form: each input enters through a smooth curve (`pygam.LinearGAM`) instead of a straight line; the hour is one cyclic curve and the three ECMWF bias terms stay linear. Section 4 needs `pygam` installed to load the saved models.

In [ ]:
import hashlib
import pickle
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display

pd.set_option("display.width", 170)
plt.rcParams.update({"figure.dpi": 110, "axes.spines.top": False, "axes.spines.right": False})

current_dir = Path.cwd().resolve()
PROJECT_ROOT = next(
    folder for folder in [current_dir, *current_dir.parents]
    if (folder / "notebooks").is_dir() and (folder / "requirements.txt").is_file()
)
RESULTS = PROJECT_ROOT / "scripts" / "results"
MODELS_FILE = PROJECT_ROOT / "models" / "linear_gam.pkl"
FORECAST_FILE = PROJECT_ROOT / "reports" / "forecast_2026-09-17_to_2026-09-30_linear_gam.csv"
HASH_FILE = RESULTS / "final_forecast_linear_gam.sha256"
FIG_DIR = PROJECT_ROOT / "reports" / "figures"

METHODS = ["Persistence", "Historical average", "ECMWF raw", "ECMWF lagged mean", "Linear regression", "Linear GAM"]
STYLES = {"Persistence": ("#bc4b51", "--"), "Historical average": ("#2a6f97", "--"), "ECMWF raw": ("#8d99ae", "-"),
          "ECMWF lagged mean": ("#f4a259", "-"), "Linear regression": ("black", "-"), "Linear GAM": ("#2a9d8f", "-")}


def result(name):
    path = RESULTS / name
    if not path.exists():
        raise FileNotFoundError(f"{path.relative_to(PROJECT_ROOT)} is missing: run scripts/linear_gam.py "
                                "(and scripts/linear_gam_test_evaluation.py for the test files) first")
    return path


def save_fig(fig, name):
    FIG_DIR.mkdir(parents=True, exist_ok=True)
    fig.savefig(FIG_DIR / f"06_{name}.png", dpi=150, bbox_inches="tight")

## 1. Validation: the two folds

Each fold trains only on targets before its first day, then all methods are scored on the same hours (as in `04`).

In [ ]:
scores = pd.read_csv(result("linear_gam_validation_scores.csv"), header=[0, 1], index_col=0)
display(scores.round(2))
mae = scores.xs("MAE", axis=1, level=1)
display(pd.DataFrame({m: 100 * (1 - mae.loc["Linear GAM"] / mae.loc[m]) for m in METHODS[:-1]}).round(0)
        .add_prefix("% lower MAE than "))

by_day = pd.read_csv(result("linear_gam_validation_mae_by_day.csv"), index_col=["fold", "lead_day"])
folds = list(by_day.index.unique("fold"))
fig, axes = plt.subplots(1, len(folds), figsize=(12, 3.6), sharey=True)
for ax, fold in zip(axes, folds):
    table = by_day.loc[fold]
    for method, (color, line) in STYLES.items():
        ax.plot(table.index, table[method], color=color, linestyle=line, marker="o", markersize=3, label=method)
    ax.set(title=fold, xlabel="Forecast day", xticks=range(1, 15))
axes[0].set(ylabel="MAE (deg C)", ylim=(0, None))
axes[0].legend(fontsize=8)
fig.tight_layout()
save_fig(fig, "validation_mae_by_day")
plt.show()

## 2. Extra check: six 2026 windows

Six forecasts set up exactly like the real task (issued at 11pm, 14 days from 12am), each from models re-trained on data before the window.

In [ ]:
windows = pd.read_csv(result("linear_gam_windows_mae.csv"), index_col=0)
display(windows.round(2))

shown = ["ECMWF raw", "Linear regression", "Linear GAM"]
names = [c for c in windows.columns if c != "Average"]
fig, ax = plt.subplots(figsize=(9, 3.4))
x = np.arange(len(names))
for k, method in enumerate(shown):
    ax.bar(x + (k - 1) * 0.27, windows.loc[method, names], width=0.27, color=STYLES[method][0], label=method)
ax.set_xticks(x, names)
ax.set(ylabel="MAE (deg C)", title="MAE of each 2026 window")
ax.legend(fontsize=8)
fig.tight_layout()
save_fig(fig, "windows_mae")
plt.show()

## 3. GAM vs. linear regression, hour by hour

From the validation errors: where in the day the two models differ, and how often the GAM is the closer of the two.

In [ ]:
errors = pd.read_csv(result("linear_gam_validation_errors.csv.gz"))
fold_errors = errors[errors["period"].isin(folds)]
shown = ["ECMWF raw", "Linear regression", "Linear GAM"]

fig, axes = plt.subplots(1, len(folds), figsize=(12, 3.4), sharey=True)
for ax, fold in zip(axes, folds):
    part = fold_errors[fold_errors["period"].eq(fold)]
    by_hour = part[shown].abs().groupby(part["target_hour"]).mean()
    for method in shown:
        color, line = STYLES[method]
        ax.plot(by_hour.index, by_hour[method], color=color, linestyle=line, marker="o", markersize=3, label=method)
    ax.set(title=fold, xlabel="Local hour", xticks=range(0, 24, 3))
axes[0].set(ylabel="MAE (deg C)")
axes[0].legend(fontsize=8)
fig.tight_layout()
save_fig(fig, "mae_by_hour")
plt.show()

closer = (fold_errors["Linear GAM"].abs() < fold_errors["Linear regression"].abs())
display(pd.DataFrame({
    "MAE difference (GAM - LR)": fold_errors.groupby("period").apply(
        lambda e: e["Linear GAM"].abs().mean() - e["Linear regression"].abs().mean()),
    "share of hours GAM is closer": closer.groupby(fold_errors["period"]).mean(),
}).round(3))

## 4. What the final GAM learned

**Partial dependence**: each input's contribution to the predicted difference from the historical average (deg C), with a 95% band, for three forecast days. A straight line means the linear regression's form was already right for that input; bends are what the GAM adds.

In [ ]:
saved = pickle.loads(MODELS_FILE.read_bytes())  # needs pygam installed
gam_models, gam_features = saved["Linear GAM"], saved["gam_features"]

SHOW_DAYS = (1, 7, 14)
terms = [i for i, term in enumerate(gam_models[1].terms) if not term.isintercept]
fig, axes = plt.subplots(len(SHOW_DAYS), len(terms), figsize=(2.3 * len(terms), 2.2 * len(SHOW_DAYS)), sharey="row")
for row, day in zip(axes, SHOW_DAYS):
    gam = gam_models[day]
    for ax, i in zip(row, terms):
        grid = gam.generate_X_grid(term=i)
        effect, band = gam.partial_dependence(term=i, X=grid, width=0.95)
        x = grid[:, gam.terms[i].feature]
        ax.plot(x, effect, color=STYLES["Linear GAM"][0])
        ax.fill_between(x, band[:, 0], band[:, 1], color=STYLES["Linear GAM"][0], alpha=0.15, lw=0)
        ax.axhline(0, color="grey", lw=0.5)
        if day == SHOW_DAYS[0]:
            ax.set_title(gam_features[gam.terms[i].feature], fontsize=8)
    row[0].set_ylabel(f"day {day}\ndeg C")
fig.tight_layout()
save_fig(fig, "partial_dependence")
plt.show()

**How much each model follows ECMWF** (1 = follow ECMWF, 0 = stay on the historical average). Linear regression: its two ECMWF coefficients added up, as in `04` Section 5.1. GAM: the average change of its prediction when both ECMWF inputs rise by 1 deg C. Below: the smoothing chosen for each forecast day (higher effective dof = wigglier curves).

In [ ]:
weight = pd.read_csv(result("linear_gam_ecmwf_weight.csv"), index_col="forecast_day")
fig, ax = plt.subplots(figsize=(9, 3.4))
for method in ["Linear regression", "Linear GAM"]:
    ax.plot(weight.index, weight[method], marker="o", color=STYLES[method][0], label=method)
ax.axhline(0, color="grey", linewidth=0.6)
ax.set(title="How much the final models follow ECMWF", xlabel="Forecast day", ylabel="Weight on ECMWF", xticks=range(1, 15))
ax.legend(fontsize=8)
fig.tight_layout()
save_fig(fig, "weight_on_ecmwf")
plt.show()

display(pd.read_csv(result("linear_gam_smoothing.csv"), index_col="forecast_day").T.round(3))

## 5. Forecast for Sep 17–30 and the test period

The frozen forecast is checked against the SHA-256 recorded by `linear_gam.py`. The test results come from `linear_gam_test_evaluation.py` (run once, after freezing); nothing here goes back into the model.

In [ ]:
assert hashlib.sha256(FORECAST_FILE.read_bytes()).hexdigest() == HASH_FILE.read_text().split()[0], "forecast changed since it was frozen"
print(HASH_FILE.read_text().splitlines()[1])

hourly = pd.read_csv(result("linear_gam_test_hourly.csv"), index_col="hour_local")
test_scores = pd.read_csv(result("linear_gam_test_scores.csv"), index_col=0)
test_by_day = pd.read_csv(result("linear_gam_test_mae_by_day.csv"), index_col=0)
print(f"Scored hours: {hourly['observed'].notna().sum()} of {len(hourly)}")
display(test_scores.round(2))

when = pd.to_datetime(hourly.index)
fig, axes = plt.subplots(2, 1, figsize=(11, 6.2), gridspec_kw={"height_ratios": [1.6, 1]})
axes[0].plot(when, hourly["observed"], color="#bc4b51", linewidth=1.8, label="observed")
axes[0].plot(when, hourly["Linear regression"], color="black", linewidth=1.1, label="linear regression")
axes[0].plot(when, hourly["Linear GAM"], color=STYLES["Linear GAM"][0], linewidth=1.4, label="linear GAM")
axes[0].plot(when, hourly["Historical average"], color="#2a6f97", linestyle="--", linewidth=1, label="historical average")
axes[0].set(title="Sep 17-30, 2026 at RDU: forecasts issued 11pm Sep 16 vs. what happened", ylabel="deg C")
axes[0].legend(fontsize=8)
for method, (color, line) in STYLES.items():
    axes[1].plot(test_by_day.columns.astype(int), test_by_day.loc[method], color=color, linestyle=line, marker="o", markersize=3, label=method)
axes[1].set(title="MAE by forecast day (test window)", xlabel="Forecast day (1 = Sep 17)", ylabel="MAE (deg C)", ylim=(0, None))
axes[1].set_xticks(range(1, 15))
axes[1].legend(fontsize=7, ncol=3)
fig.tight_layout()
save_fig(fig, "test_forecast_vs_observed")
plt.show()